# Análise de Dados
---
Respostas às perguntas de negócio.

### P1. Como evoluíram os preços unitários registrados no BPS entre 2024 e 2026? 

In [0]:
SELECT
    CAST(ano_compra AS STRING) AS ano_compra, -- Transformação do ano para string para facilitar a visualização no gráfico
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT sk_produto) AS qtd_catmats,
    ROUND(AVG(vl_preco_unitario), 2) AS preco_medio,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS preco_mediano,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3
FROM mvp_puc_engenhariadedados.gold.fato_compra
WHERE vl_preco_unitario IS NOT NULL
  AND vl_preco_unitario > 0
GROUP BY ano_compra
ORDER BY ano_compra;

ano_compra,qtd_registros,qtd_catmats,preco_medio,preco_mediano,q1,q3
2024,28958,4514,684.77,1.86,0.33,7.41
2025,34434,5205,288.82,2.11,0.38,8.24
2026,11411,3113,256.12,1.88,0.32,7.50


Databricks visualization. Run in Databricks to view.

### P2. Quais itens (CATMATs) apresentam maior dispersão de preços no período? 

Para identificação dos CATMATs com maior dispersão de preços, foram utilizadas duas medidas complementares: o Intervalo Interquartil (IQR), que representa a dispersão entre o primeiro e o terceiro quartis, e o Coeficiente de Variação (CV), que relaciona o desvio-padrão à média.   

In [0]:
SELECT
    co_catmat,
    ds_item,
    COUNT(*) AS qtd_compras,
    ROUND(MIN(vl_preco_unitario), 2) AS preco_minimo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS mediana,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3,
    ROUND(MAX(vl_preco_unitario), 2) AS preco_maximo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000)- PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS iqr
FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis
GROUP BY co_catmat, ds_item
ORDER BY iqr DESC
LIMIT 20;

co_catmat,ds_item,qtd_compras,preco_minimo,q1,mediana,q3,preco_maximo,iqr
458500,"ALECTINIBE, CONCENTRAÇÃO:150 MG",5,105.00,105.54,105.67,36059.47,36059.47,35953.93
480036,"ISATUXIMABE, CONCENTRAÇÃO:20 MG/ML, FORMA FARMACÊUTICA:INJETÁVEL",6,2597.67,2678.34,2712.03,13560.14,13646.56,10881.80
400563,"USTEQUINUMABE, CONCENTRAÇÃO:90 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",17,6395.62,11204.36,20690.00,21900.00,32795.60,10695.64
455395,"USTEQUINUMABE, CONCENTRAÇÃO:5 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",12,10920.40,21840.84,26314.26,27127.00,38530.00,5286.16
441461,"DARATUMUMABE, CONCENTRAÇÃO:20 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",11,1598.89,1598.90,6395.61,6395.64,6561.93,4796.74
436778,"NIVOLUMABE, CONCENTRAÇÃO:10 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",20,2718.79,2841.16,7102.91,7287.58,16728.90,4446.42
412718,"ICATIBANTO ACETATO, CONCENTRAÇÃO:10 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",8,2507.40,3030.00,4953.60,6882.32,7192.03,3852.32
357119,"BETA-AGALSIDASE, CONCENTRAÇÃO:35 MG, FORMA FARMACÊUTICA:PÓ LIÓFILO P/ INJETÁVEL",11,13102.89,13358.56,13961.27,17023.78,17466.40,3665.22
390008,"CETUXIMABE, CONCENTRAÇAO:5 MG/ML, FORMA FARMACEUTICA:SOLUÇÃO INJETÁVEL",10,813.11,813.12,813.12,4065.58,4171.28,3252.46
290058,"ADALIMUMABE, CONCENTRAÇÃO:40 MG, APRESENTAÇÃO:SOLUÇÃO INJETÁVEL",17,390.00,490.60,1035.52,3533.00,3668.33,3042.40


In [0]:
SELECT
    co_catmat,
    ds_item,
    COUNT(*) AS qtd_compras,
    ROUND(AVG(vl_preco_unitario), 2) AS preco_medio,
    ROUND(STDDEV(vl_preco_unitario), 2) AS desvio_padrao,
    ROUND(
        100.0 * STDDEV(vl_preco_unitario)
        / NULLIF(AVG(vl_preco_unitario), 0),
        2
    ) AS cv_percentual,
    ROUND(MIN(vl_preco_unitario), 2) AS preco_minimo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS mediana,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3,
    ROUND(MAX(vl_preco_unitario), 2) AS preco_maximo,
    ROUND(
        PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000)
        - PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000),
        2
    ) AS iqr
FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis
GROUP BY co_catmat, ds_item
ORDER BY cv_percentual DESC
LIMIT 20;

co_catmat,ds_item,qtd_compras,preco_medio,desvio_padrao,cv_percentual,preco_minimo,q1,mediana,q3,preco_maximo,iqr
271089,"AMOXICILINA, CONCENTRAÇÃO:500MG",224,231.57,3410.08,1472.61,0.04,0.19,0.21,0.27,51038.16,0.08
267517,"ATENOLOL, DOSAGEM:50 MG",190,0.55,6.53,1191.81,0.02,0.04,0.05,0.06,90.00,0.02
268252,"DIPIRONA SÓDICA, DOSAGEM:500 MG/ML, APRESENTAÇÃO:SOLUÇÃO INJETÁVEL",176,6.69,77.58,1159.83,0.04,0.51,0.72,0.89,1030.00,0.38
267741,"PREDNISONA, DOSAGEM:5 MG",160,1.43,16.6,1157.61,0.02,0.06,0.06,0.07,210.00,0.01
268856,"LOSARTANA POTÁSSICA, DOSAGEM:50 MG",185,0.68,7.72,1137.3,0.01,0.04,0.04,0.05,105.00,0.02
383750,"LACTULOSE, CONCENTRAÇÃO:667 MG/ML, FORMA FARMACEUTICA:XAROPE",134,91.44,995.15,1088.33,0.05,3.70,4.39,5.47,11525.00,1.77
267743,"PREDNISONA, DOSAGEM:20 MG",170,8.24,89.34,1083.69,0.04,0.14,0.16,0.20,1149.00,0.06
267613,"CAPTOPRIL, CONCENTRAÇÃO:25 MG",121,0.44,4.36,995.56,0.02,0.02,0.03,0.04,48.00,0.02
270119,"CLONAZEPAM, DOSAGEM:2 MG",205,0.63,6.27,988.08,0.02,0.04,0.05,0.06,86.70,0.02
268255,"EPINEFRINA, DOSAGEM:1MG/ML, USO:SOLUÇÃO INJETÁVEL",156,8.90,87.66,984.57,0.26,0.90,1.06,1.23,1090.00,0.33


### P3. Quais compras apresentam preços potencialmente atípicos em relação às demais compras do mesmo CATMAT? 

In [0]:
SELECT
    -- Total de registros potencialmente atípicos
    COUNT(*) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite
FROM mvp_puc_engenhariadedados.gold.vw_potenciais_outliers;

qtd_potencialmente_atipicos,qtd_abaixo_limite,qtd_acima_limite,percentual_abaixo_limite,percentual_acima_limite
7807,1181,6626,15.13,84.87


In [0]:
SELECT
    co_seq_bps,
    ano_compra,
    data_completa,
    co_catmat,
    ds_item,
    ROUND(vl_preco_unitario, 2) AS vl_preco_unitario,
    ROUND(limite_inferior, 2) AS limite_inferior,
    ROUND(limite_superior, 2) AS limite_superior,
    CASE
        WHEN tipo_atipicidade = 'Abaixo do limite inferior'
            THEN ROUND(limite_inferior - vl_preco_unitario, 2)
        WHEN tipo_atipicidade = 'Acima do limite superior'
            THEN ROUND(vl_preco_unitario - limite_superior, 2)
    END AS distancia_do_limite,
    tipo_atipicidade,
    tp_compra,
    modalidade,
    sg_uf,
    no_municipio,
    no_instituicao,
    no_fornecedor
FROM mvp_puc_engenhariadedados.gold.vw_potenciais_outliers
ORDER BY distancia_do_limite DESC;

-- NOTA: É importante observar que a tabela output a seguir apresenta 6661+ linhas, ao invés de 7807, pois o resultado foi truncado devido à limitação de tamanho pela visualização do Databricks Free Edition. Entretanto, ao realizar o download do resultado em formato arquivo CSV foi possível observar o resultado completo e conforme o esperado.

co_seq_bps,ano_compra,data_completa,co_catmat,ds_item,vl_preco_unitario,limite_inferior,limite_superior,distancia_do_limite,tipo_atipicidade,tp_compra,modalidade,sg_uf,no_municipio,no_instituicao,no_fornecedor
17628923,2024,2024-11-13,606824,"TEZEPELUMABE, CONCENTRAÇÃO:110 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL, CARACTERÍSTICAS ADICIONAIS:SERINGA PREENCHIDA",8191720.00,7480.74,8147.34,8183572.66,Acima do limite superior,JUDICIAL,Pregão,RJ,RIO DE JANEIRO,SECRETARIA DE ESTADO DE SAUDE - SES,ASTRAZENECA DO BRASIL LTDA.
17664330,2026,2026-01-02,270895,"CARBONATO DE CÁLCIO, DOSAGEM:500MG DE CÁLCIO",1149000.00,-0.73,1.37,1148998.63,Acima do limite superior,ADMINISTRATIVA,Pregão,RS,MONTENEGRO,CONSORCIO INTERMUNICIPAL DO VALE DO RIO CAI (CIS-CAI),ALTERMED MATERIAL MEDICO HOSPITALAR LTDA
17830908,2025,2025-12-11,305725,"OCTREOTIDA, DOSAGEM:0,1 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL",520000.00,-5.64,147.22,519852.78,Acima do limite superior,ADMINISTRATIVA,Pregão,SP,SAO PAULO,SECRETARIA DE ESTADO DA SAUDE,PORTAL LTDA
17709621,2024,2024-09-27,439214,"CUBA USO HOSPITALAR, MATERIAL:AÇO INOX, FORMATO:TIPO RIM, CAPACIDADE:CERCA DE 700 ML",362750.00,0.15,124.07,362625.93,Acima do limite superior,ADMINISTRATIVA,Pregão,SE,CAMPO DO BRITO,FUNDO MUNICIPAL DE SAUDE,ESTRELA COMERCIO DE PRODUTOS MEDICOS E HOSPITALARES LTDA
17710588,2025,2025-12-01,272815,"PENICILAMINA, DOSAGEM:250 MG",294400.00,27.59,30.55,294369.45,Acima do limite superior,ADMINISTRATIVA,Pregão,PR,CURITIBA,SECRETARIA DE ESTADO DA SAUDE,AGILLE COMERCIO DE MEDICAMENTOS LTDA
17611652,2024,2024-11-25,439259,"AXITINIBE, CONCENTRAÇÃO:5 MG",253090.00,172.80,386.92,252703.08,Acima do limite superior,JUDICIAL,Pregão,RJ,RIO DE JANEIRO,SECRETARIA DE ESTADO DE SAUDE - SES,DISTRIBUICAO DE MEDICAMENTOS PAMED LTDA
17630416,2024,2024-09-27,437552,"CANETA ALTA ROTAÇÃO, MATERIAL ROLAMENTO:ROLAMENTO CERÂMICA, VELOCIDADE MÁXIMA:VELOCIDADE MÁXIMA MENOR OU IGUAL 400.000 RPM, REFRIGERAÇÃO:3 OU MAIS FUROS, TROCA DE BROCAS:BOTÃO DE PRESSÃO(PB), TIPO CONEXÃO:CONEXÃO 2 FUROS, TIPO CABEÇA:CABEÇA PADRÃO",249990.00,-49.97,669.95,249320.05,Acima do limite superior,ADMINISTRATIVA,Pregão,SE,CAMPO DO BRITO,FUNDO MUNICIPAL DE SAUDE,"D-X INDUSTRIA, COMERCIO, IMPORTACAO E EXPORTACAO LTDA"
17680365,2024,2024-03-27,433445,"ACESSÓRIO BOMBA INSULINA, TIPO ACESSÓRIO:RESERVATÓRIO, MATERIAL:POLIPROPILENO TRANSPARENTE, COMPONENTE:TIPO SERINGA CERCA 3 ML",202000.00,-1548.00,3488.80,198511.20,Acima do limite superior,JUDICIAL,Inexigibilidade de Licitação,SP,GUARARAPES,MUNICIPIO DE GUARARAPES,MEDTRONIC COMERCIAL LTDA
17724757,2025,2025-01-08,356051,"ÁCIDO ZOLEDRÔNICO, CONCENTRAÇÃO:50 MCG/ML, FORMA FARMACEUTICA:SOLUÇÃO INJETÁVEL",199000.00,-135.23,801.14,198198.86,Acima do limite superior,ADMINISTRATIVA,Pregão,SP,SAO PAULO,SECRETARIA DE ESTADO DA SAUDE,SP HOSPITALAR LTDA
17638698,2025,2025-06-04,480015,"ROMOSOZUMABE, CONCENTRAÇÃO:90 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL, ADICIONAL:SERINGA PREENCHIDA",161765.00,-342.66,2848.43,158916.57,Acima do limite superior,JUDICIAL,Dispensa de Licitação,PB,JOAO PESSOA,SECRETARIA DE ESTADO DA SAUDE - SES,ONCOVIT DISTRIBUIDORA DE MEDICAMENTOS LTDA


### P4a. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem ao longo do tempo, por ano?

In [0]:
SELECT
    CAST(e.ano_compra AS STRING) AS ano_compra, -- Transformação do ano para string para facilitar a visualização no gráfico

    -- Total de registros elegíveis no ano
    COUNT(*) AS qtd_registros_elegiveis,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.ano_compra
ORDER BY e.ano_compra;

ano_compra,qtd_registros_elegiveis,qtd_potencialmente_atipicos,qtd_abaixo_limite,qtd_acima_limite,percentual_potencialmente_atipicos,percentual_abaixo_limite,percentual_acima_limite
2024,26071,3179,401,2778,12.19,1.54,10.66
2025,30563,3585,529,3056,11.73,1.73,10.00
2026,10178,1043,251,792,10.25,2.47,7.78


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### P4b. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por tipo de compra?

In [0]:
SELECT
    e.tp_compra,

    -- Total de registros elegíveis por tipo de compra
    COUNT(*) AS qtd_registros_elegiveis,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.tp_compra
ORDER BY percentual_potencialmente_atipicos DESC;

tp_compra,qtd_registros_elegiveis,qtd_potencialmente_atipicos,qtd_abaixo_limite,qtd_acima_limite,percentual_potencialmente_atipicos,percentual_abaixo_limite,percentual_acima_limite
JUDICIAL,4832,971,122,849,20.10,2.52,17.57
ADMINISTRATIVA,61980,6836,1059,5777,11.03,1.71,9.32


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### 4c. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por modalidade de compra?

In [0]:
SELECT
    e.modalidade,

    -- Total de registros elegíveis por modalidade
    COUNT(*) AS qtd_registros_elegiveis,

    -- Representatividade da modalidade quando comparada com o total de registros elegíveis
    CASE
        WHEN COUNT(*) < 100
            THEN 'Baixa'
        ELSE 'Adequada'
    END AS representatividade,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.modalidade
ORDER BY percentual_potencialmente_atipicos DESC;

modalidade,qtd_registros_elegiveis,representatividade,qtd_potencialmente_atipicos,qtd_abaixo_limite,qtd_acima_limite,percentual_potencialmente_atipicos,percentual_abaixo_limite,percentual_acima_limite
Concorrência,1,Baixa,1,0,1,100.00,0.00,100.00
Dispensa de Licitação,2413,Adequada,638,39,599,26.44,1.62,24.82
Pregão,53199,Adequada,6059,952,5107,11.39,1.79,9.60
Inexigibilidade de Licitação,19,Baixa,2,0,2,10.53,0.00,10.53
Concurso,10,Baixa,1,0,1,10.00,0.00,10.00
Registro de Preços,11142,Adequada,1105,189,916,9.92,1.70,8.22
Leilão,27,Baixa,1,1,0,3.70,3.70,0.00
Diálogo Competitivo,1,Baixa,0,0,0,0.00,0.00,0.00


### P4d. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por fornecedor, entre aqueles com volume financeiro relevante?

In [0]:
SELECT
    e.cnpj_fornecedor,
    e.no_fornecedor,

    -- Total de registros elegíveis por fornecedor
    COUNT(*) AS qtd_registros_elegiveis,

    -- Representatividade do fornecedor quando comparada com o total de registros elegíveis
    CASE
        WHEN COUNT(*) < 100
            THEN 'Baixa'
        ELSE 'Adequada'
    END AS representatividade,

    -- Valor total das vendas elegíveis
    ROUND(
        SUM(e.vl_preco_total),
        2
    ) AS valor_total_vendas,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.cnpj_fornecedor, e.no_fornecedor
HAVING SUM(e.vl_preco_total) >= 1000000
ORDER BY percentual_potencialmente_atipicos DESC;

cnpj_fornecedor,no_fornecedor,qtd_registros_elegiveis,representatividade,valor_total_vendas,qtd_potencialmente_atipicos,qtd_abaixo_limite,qtd_acima_limite,percentual_potencialmente_atipicos,percentual_abaixo_limite,percentual_acima_limite
01772798000233,MEDTRONIC COMERCIAL LTDA,1,Baixa,1818000.00,1,0,1,100.00,0.00,100.00
07090403000118,TRIMEDCALL COMERCIO DE MATERIAIS MEDICOS E HOSPITALARES LTDA,2,Baixa,2557468.00,2,2,0,100.00,100.00,0.00
33009945002339,PRODUTOS ROCHE QUIMICOS E FARMACEUTICOS S A,1,Baixa,17276477.40,1,0,1,100.00,0.00,100.00
02228938000199,"D-X INDUSTRIA, COMERCIO, IMPORTACAO E EXPORTACAO LTDA",1,Baixa,14999400.00,1,0,1,100.00,0.00,100.00
37140339000101,A.L.V. DISTRIBUIDORA DE PRODUTOS HOSPITALARES LTDA,2,Baixa,34512845.08,2,0,2,100.00,0.00,100.00
08160290000142,"FARMAGUEDES COMERCIO DE PRODUTOS FARMACEUTICOS, MEDICOS E HOSPITALARES LTDA",168,Adequada,1252238.50,144,0,144,85.71,0.00,85.71
03894963000174,"SAO MARCOS DISTRIBUIDORA DE MEDICAMENTOS, EQUIPAMENTOS E MATERIAIS HOSPITALARES E ODONTOLOGICOS LTDA",626,Adequada,2555835.83,533,0,533,85.14,0.00,85.14
04389760000193,PRADO PHARMA LTDA,185,Adequada,2338821.16,153,0,153,82.70,0.00,82.70
02335871000191,DISPROFARMA COMERCIO LTDA,133,Adequada,1847658.50,97,0,97,72.93,0.00,72.93
11311773000105,OKEY-MED DISTRIBUIDORA DE MEDICAMENTOS HOSPITALARES ODONTOLOGICOS IMPORTACOES E EXPORTACOES LTDA,7,Baixa,1132021.50,5,1,4,71.43,14.29,57.14
